# S4 pilot — local generation on Kaggle T4

**This notebook is a RUNNER only.** It clones the repo, installs, and calls scripts.
No logic lives here — `CLAUDE.md`: *"If logic lives in a notebook cell, it cannot
enter the paper."*

⛔ **The pilot is NOT A RESULT.** It selects a generator; it measures nothing.

## The two arms

| arm | model | difference |
|---|---|---|
| A | `google/gemma-3-12b-it` | general multilingual |
| B | `md-nishat-008/TigerLLM-9B-it` | **same base, Bangla-adapted** |

Verified from `config.json`: both are `Gemma3ForCausalLM`, Gemma-3 family,
vocab 262144. **One variable: Bangla adaptation.**

⚠️ The TigerLLM paper (arXiv 2503.10995) says its 1B is built on **LLaMA-3.2**.
The uploaded weights are **Gemma-3**. The paper's benchmark table therefore does
not describe these weights, and no claim here rests on it.

**Settings: GPU T4 x2 · Internet ON · attach `bn_clean.csv` as a Dataset.**


## 0. Smoke test FIRST — before anything else

Gemma-3 ships **bf16** weights and a T4 (Turing, sm_75) has no bf16. Casting to
fp16 is a known risk for this family. So the first thing this notebook does is
generate one sample and **print it** — two of the three bugs found on 2026-08-11
were caught by reading a rendered artifact, not by a test.

**If the output is empty, English, or garbage, stop here.**


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install -U transformers accelerate 2>&1 | tail -2


### HF token — needed for `gemma-3-12b-it` (arm A), which is gated

Accept the licence once at **huggingface.co/google/gemma-3-12b-it**, make a
**Read** token, then add it in **Add-ons → Secrets** as `HF_TOKEN`.

⚠️ Never paste the token into a cell — a shared or public notebook carries it.


In [ ]:
from kaggle_secrets import UserSecretsClient
import os
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')))


### Weights — where they come from (REVISED 2026-08-12, after ENOSPC)

The first plan — `snapshot_download(local_dir=/kaggle/working/...)` then save
as a Dataset — **failed with `No space left on device` and could never have
worked**: `local_dir` keeps a second copy in the HF cache (~2× disk), and
`/kaggle/working` caps at ~19.5 GB against ~24 GB of weights. Logged as a
deviation in `protocol.md`.

**Arm A (`google/gemma-3-12b-it`): attach from Kaggle Models — no download.**
Add Input → Models → search *gemma 3* → `google/gemma-3` →
variation **transformers / gemma-3-12b-it**. Read-only mount, zero disk cost.

**Arm B (`md-nishat-008/TigerLLM-9B-it`): plain cache download, no `local_dir`.**
One copy in `/root/.cache`, which fits. It re-downloads each session — a few
minutes on Kaggle's network — and that is cheaper than fighting the disk cap.


In [ ]:
# If a previous attempt filled the disk, clear it first.
!rm -rf /kaggle/working/models
!df -h /kaggle/working /root | head -3


In [ ]:
# Arm B -> HF cache (single copy). Arm A comes from the Kaggle Models mount.
from huggingface_hub import snapshot_download
print('cached:', snapshot_download('md-nishat-008/TigerLLM-9B-it'))


In [ ]:
# Arm A's mount path -- confirm it exists and note it: it is the value of
# --model-path in the pilot cell below. Kaggle has mounted models under both
# /kaggle/input/<slug>/ and /kaggle/input/models/<owner>/<slug>/ depending on
# platform version (observed 2026-08-12), so both layouts are searched.
import glob
hits = sorted(glob.glob('/kaggle/input/gemma-3/transformers/*12b-it*/*') +
              glob.glob('/kaggle/input/models/google/gemma-3/transformers/*12b-it*/*'))
assert hits, 'gemma-3-12b-it not attached: Add Input -> Models -> google/gemma-3 -> transformers/gemma-3-12b-it'
GEMMA_PATH = hits[-1]
print('GEMMA_PATH =', GEMMA_PATH)


In [ ]:
import torch, time
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 4-bit: 12.19B in fp16 is ~24 GB and a T4 has 16 GB. IDENTICAL for both arms,
# or the comparison measures the quantiser rather than the model.
q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                       bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True)

prompt = 'তুমি একজন সাধারণ বাংলাদেশি দর্শক। একটি সিনেমা নিয়ে ছোট একটি মন্তব্য লেখো।'

# Smoke-test BOTH arms: arm B from the HF cache, arm A from the Kaggle mount.
for name, src in [('TigerLLM-9B-it', 'md-nishat-008/TigerLLM-9B-it'),
                  ('gemma-3-12b-it', GEMMA_PATH)]:
    tok = AutoTokenizer.from_pretrained(src)
    m = AutoModelForCausalLM.from_pretrained(src, quantization_config=q, device_map='auto')
    text = tok.apply_chat_template([{'role':'user','content':prompt}],
                                   tokenize=False, add_generation_prompt=True)
    enc = tok(text, return_tensors='pt', add_special_tokens=False).to(m.device)
    t0 = time.time()
    out = m.generate(**enc, do_sample=True, temperature=0.8, top_p=0.9, max_new_tokens=80)
    print(f'--- {name}: {time.time()-t0:.1f}s, prompt tokens {enc["input_ids"].shape[1]}')
    print(tok.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True))
    del m; torch.cuda.empty_cache()


### Read the output above before continuing.

- **Bangla, coherent, comment-like** → continue.
- **Empty / `nan` / repeated tokens** → fp16 overflow. Try `bfloat16` (will fail on
  T4) or fall back to the API path. Record the failure; do not work around it.
- **English** → the chat template or the model is not doing what we assume.


## 1. Repo, data, index


In [ ]:
!git clone -q https://github.com/alphapie77/BSc_Thesis.git repo
%cd repo
!pip -q install chromadb sentence-transformers pyyaml 2>&1 | tail -2


In [ ]:
# bn_clean.csv is gitignored (size + licence), so it arrives as a Kaggle Dataset.
!mkdir -p data/cleaned
!cp /kaggle/input/datasets/alphapie77/bn-clean/bn_clean.csv data/cleaned/bn_clean.csv
!ls -l data/cleaned/


In [ ]:
# Rebuild the R1-only index here. It refuses to run if an R2 or Gold-300 id
# reaches it (inviolable rules 4 and 5), checked twice by two mechanisms.
!python src/agents/build_index.py --config configs/s4_index.yaml


**Expect `886` rows and a digest starting `85fc2d7d`.** A different digest means
different rows went in, and nothing downstream is comparable to the local run.


## 2. Dry run — print the real prompt

No generation. Reads the actual retrieval and prints the full prompt for both
levels and both language arms.


In [ ]:
!python src/agents/run_pilot.py --config configs/s4_pilot_local.yaml --dry-run 2>&1 | head -80


## 3. The pilot

Resumable: every generation is appended to JSONL as it completes, and a re-run
skips what is on disk. A 12-hour session cap cannot lose the run.


In [ ]:
# One line, no backslash continuation: IPython expands $VAR only on the FIRST
# line of a ! cell, so the continued form silently passed an EMPTY path and the
# run aborted at the grid (2026-08-15). Exporting to the environment first makes
# the expansion the shell's job, which does not depend on cell layout.
import os; os.environ['GEMMA_PATH'] = GEMMA_PATH
!python src/agents/run_pilot.py --config configs/s4_pilot_local.yaml --model-path arm_a=$GEMMA_PATH


## 4. Save the outputs back

⚠️ **Kaggle resets disk between sessions.** The JSONL is the reproducibility
artifact — `2601.17768` means these generations cannot be regenerated — so it must
leave the notebook. Download `/kaggle/working/` and commit it to the repo.


In [ ]:
# Filenames come from configs/s4_pilot_local.yaml `outputs:` -- the local run
# writes pilot_s4_local_*, deliberately separate from the retired Groq archive
# (pilot_s4_generations.jsonl). No 2>/dev/null: a failed copy must be SEEN --
# the JSONL cannot be regenerated, so a silent copy failure loses the run.
!cp results/pilot_s4_local_generations.jsonl /kaggle/working/
!cp results/pilot_s4_local_model_choice.md results/pilot_s4_local_model_choice.json /kaggle/working/
!python src/common/env_snapshot.py && cp results/env_snapshot.json /kaggle/working/env_snapshot_s4_kaggle.json
!ls -lh /kaggle/working/
